In [ ]:
import pandas as pd 

In [23]:
df_os = pd.read_csv("../data/raw/openalex/openalex_sri_lanka_works.csv")
print(df_os.shape)

df_cr=pd.read_csv("../data/processed/crossref/crossref_sri_lanka_works.csv")
print(df_cr.shape)

/tmp/ipykernel_164907/712480020.py:1: DtypeWarning: Columns (0: is_retracted) have mixed types. Specify dtype option on import or set low_memory=False.
  df_os = pd.read_csv("../data/raw/openalex/openalex_sri_lanka_works.csv")


(59942, 44)
(16959, 34)


/tmp/ipykernel_164907/712480020.py:4: DtypeWarning: Columns (0: volume, 1: publisher-location, 2: event.sponsor, 3: original-title) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cr=pd.read_csv("../data/processed/crossref/crossref_sri_lanka_works.csv")


In [24]:
os_types=df_os["type"].value_counts()
cr_types=df_cr["type"].value_counts()
print(os_types)
print(cr_types)

type
article                51301
book-chapter            2825
preprint                1672
review                  1481
report                   876
letter                   445
dataset                  410
conference-paper         214
other                    158
peer-review              147
book                     126
editorial                 77
dissertation              51
reference-entry           45
software                  40
erratum                   34
paratext                  25
conference-abstract       11
retraction                 4
Name: count, dtype: int64
type
journal-article        11204
proceedings-article     5165
posted-content           590
Name: count, dtype: int64


In [25]:
KEEP_TYPES = {
    "article",
    "conference-paper",
    "preprint",
    "review",
    "letter",
    "editorial",
    "erratum",
    "retraction",
}

filtered_os=df_os[
    (df_os["type"].isin(KEEP_TYPES)) & 
    (df_os["publication_year"]>=2000)]

print(filtered_os.shape)

(51563, 44)


In [26]:
print(len(filtered_os))
print(filtered_os["publication_year"].min())
print(filtered_os["publication_year"].max())
print(filtered_os["type"].value_counts())


51563
2000.0
2026.0
type
article             47717
preprint             1667
review               1445
letter                410
conference-paper      214
editorial              73
erratum                33
retraction              4
Name: count, dtype: int64


In [27]:
print(len(df_cr))
print(df_cr["published.date-parts"].min())
print(df_cr["published.date-parts"].max())
print(df_cr["type"].value_counts())


df_cr["year"] = df_cr["issued.date-parts"].apply(
    lambda x: x[0][0] if isinstance(x, list) and len(x) > 0 else None
)

print(df_cr["year"].min())
print(df_cr["year"].max())

16959
[[2000, 1, 1]]
[[2026]]
type
journal-article        11204
proceedings-article     5165
posted-content           590
Name: count, dtype: int64
nan
nan


In [28]:
filtered_os["doi"] = filtered_os["doi"].astype(str).str.strip().str.lower()

df_cr["DOI"] =df_cr["DOI"].astype(str).str.strip().str.lower()

filtered_os["doi"] = filtered_os["doi"].str.replace("https://doi.org/", "", regex=False)


openalex_dois = set(filtered_os["doi"].dropna())

crossref_dois = set(df_cr["DOI"].dropna())

matching_dois = openalex_dois & crossref_dois

print("OpenAlex DOIs:", len(openalex_dois))
print("Crossref DOIs:", len(crossref_dois))
print("Matching DOIs:", len(matching_dois))

OpenAlex DOIs: 47246
Crossref DOIs: 16959
Matching DOIs: 8332


In [ ]:
openalex_only=openalex_dois-crossref_dois
print(len(openalex_only))



38914


{'',
 '10.1109/icter.2011.6075022',
 '10.4038/jpgim.8256',
 '10.1080/14620316.2007.11512214',
 '10.4038/cjmr.v4i1.39',
 '10.4038/tare.v10i0.1865',
 '10.1109/mercon.2018.8421980',
 '10.31357/fesympo.v24i0.4249',
 '10.6084/m9.figshare.30796555.v1',
 '10.4038/sljss.v43i1.8069',
 '10.1016/j.ghir.2017.08.006',
 '10.1080/01421590902833036',
 '10.5281/zenodo.13730166',
 '10.4161/cbt.28584',
 '10.21921/jas.v3i4.6713',
 '10.4038/cmj.v48i3.3361',
 '10.4038/jm.v8i1.7551',
 '10.4038/jnsfsr.v52i1.11403',
 '10.4038/besl.v7i1.1945',
 '10.2139/ssrn.3844228',
 '10.1093/trstmh/traf147',
 '10.1017/s0068113x10000462',
 '10.1016/j.jsamd.2023.100533',
 '10.5281/zenodo.20702795',
 '10.31357/fesympo.v0i0.1322',
 '10.1504/ijbcg.2016.082203',
 '10.1016/j.psep.2022.04.074',
 '10.31357/fesympo.v21i0.3164.g2352',
 '10.2741/e856',
 '10.1007/s12161-023-02552-y',
 '10.1186/s12891-017-1562-9',
 '10.7176/rjfa/10-22-10',
 '10.1080/14888386.2001.9712671',
 '10.1248/jhs.55.40',
 '10.1109/icdabi53623.2021.9655781',
 '10.11

In [37]:
import time
import requests
import random

In [36]:
session=requests.Session()

session.headers.update({
    "User-Agent":"ResearchLanka/1.0 (mailto:your@email.com)"}
)



In [54]:
import pandas as pd

sample_dois = pd.Series(list(openalex_only)).sample(n=1000, random_state=42).tolist()

print(len(sample_dois))
 

1000


In [55]:
found=[]
missing=[]

for i,doi in enumerate(sample_dois):
    url = f"https://api.crossref.org/works/{doi}"
    try:
        response = session.get(url, timeout=20)

        if response.status_code == 200:
            found.append(doi)
        else:
            missing.append(doi)

    except requests.RequestException:
        missing.append(doi)
    time.sleep(0.2)

    if i % 100 == 0:
        print(
            f"Processed {i}/{len(sample_dois)} | "
            f"Found: {len(found)} | "
            f"Missing: {len(missing)}"
        )


print("Total tested:", len(sample_dois))
print("Found in Crossref:", len(found))
print("Missing in Crossref:", len(missing))


Processed 0/1000 | Found: 1 | Missing: 0
Processed 100/1000 | Found: 94 | Missing: 7
Processed 200/1000 | Found: 189 | Missing: 12
Processed 300/1000 | Found: 277 | Missing: 24
Processed 400/1000 | Found: 373 | Missing: 28
Processed 500/1000 | Found: 468 | Missing: 33
Processed 600/1000 | Found: 562 | Missing: 39
Processed 700/1000 | Found: 654 | Missing: 47
Processed 800/1000 | Found: 747 | Missing: 54
Processed 900/1000 | Found: 845 | Missing: 56
Total tested: 1000
Found in Crossref: 941
Missing in Crossref: 59


In [44]:
print(found)

['10.4038/slaj.v2i2.8', '10.4038/tar.v34i1.8602', '10.15406/bbij.2021.10.00328', '10.9734/acri/2021/v21i230230', '10.1080/00038628.2002.9697499', '10.34293/education.v10i3.4489', '10.4038/nsbmjm.v2i2.29', '10.4038/sljer.v5i2.48', '10.4038/cmj.v64i4.8992', '10.11591/ijins.v1i2.438', '10.1016/j.gheart.2012.01.003', '10.1080/00103624.2011.539084', '10.4038/cmj.v48i4.3338', '10.1504/ijmmno.2017.10007737', '10.4038/tar.v34i3.8646', '10.4038/jmj.v32i1.98', '10.4038/wjm.v3i2.7442', '10.4038/kjm.v11i1.7656', '10.4038/sljog.v41i2.7888', '10.4038/jfa.v11i2.5212', '10.4038/kjm.v5i1.7504', '10.4038/jpgim.8391', '10.1038/s41598-024-73538-x', '10.4038/tar.v24i3.8005', '10.47772/ijriss.2021.5743', '10.3329/jbip.v16i1.77040', '10.2139/ssrn.1992372', '10.31357/fesympo.v0i0.1734', '10.70844/ajeer.2025.1.3', '10.12691/education-6-3-13', '10.4038/sjdem.v1i1.4192', '10.1109/mercon.2018.8421886', '10.2139/ssrn.4139352', '10.18535/ijsrm/v7i3.sh02', '10.4038/sljastats.v19i2.8021', '10.21769/bioprotoc.1261', '

In [51]:
for test_doi in found:
    matched = df_cr[df_cr["DOI"].str.contains(test_doi, na=False)]

    if not matched.empty:
        print("yes", test_doi)


In [52]:
crossref_only = crossref_dois - openalex_dois

print("Crossref only:", len(crossref_only))


Crossref only: 8627


In [53]:
print(crossref_only)

{'10.1021/acsanm.9b00430', '10.1109/icoin68469.2026.11480634', '10.1161/circ.150.suppl_1.4136294', '10.1111/mec.13751', '10.1021/acs.jnatprod.7b00838', '10.12982/cmjs.2026.055', '10.1080/09720510.2016.1228321', '10.1109/mercon52712.2021.9525635', '10.1097/mpg.0000000000002839', '10.1177/1010539515611723', '10.1002/etc.5110', '10.3390/engproc2023039017', '10.3346/jkms.2016.31.8.1190', '10.1109/ieem62345.2024.10857241', '10.1002/wat2.1394', '10.1079/searchrxiv.2026.01308', '10.1136/bmjgh-2016-000057', '10.1109/access.2025.3526666', '10.1109/hipcw63042.2024.00065', '10.65646/3rc20dmm3h0092', '10.36878/nsj20220501.01', '10.1258/cvd.2012.012019', '10.3390/electronics10131569', '10.1099/acmi.0.001137.v3', '10.1109/icosnikom48755.2019.9111585', '10.3390/civileng7010009', '10.1163/15685381-20191215', '10.1111/andr.13597', '10.1163/15718085-bja10101', '10.3390/analytica4010003', '10.1002/ird.2590', '10.1287/ijoc.2018.0812', '10.1111/ajr.12552', '10.1530/rep.1.00183', '10.1111/odi.14306', '10.12

In [ ]:
funders = df_cr[df_cr["funder"].notna()]["DOI"]
print(len(funders))
print(funders["DOI"])




3607
22                  10.1145/3505711.3505715
23       10.1109/ssitcon62437.2024.10796169
59                        10.3390/s24113386
102             10.1109/access.2025.3584890
111             10.1136/bmjopen-2024-092262
                        ...                
16954           10.1021/acs.jchemed.5c00255
16955                     10.3390/w12082209
16956              10.1021/acsomega.4c09061
16957                  10.3390/land10020218
16958                   10.1093/jtm/taaf052
Name: DOI, Length: 3607, dtype: str


In [80]:
for doi in funders:

   record = df_os[df_os["doi"].str.contains(doi, case=False, na=False)]
   if not record.empty:
       print(record["doi"])




44039    https://doi.org/10.20659/jfp.16.special_issue_79
57109    https://doi.org/10.37547/ijll/volume06issue03-44
Name: doi, dtype: str
13991     https://doi.org/10.5465/ambpp.2017.15903abstract
25490     https://doi.org/10.5465/ambpp.2017.14344abstract
27232    https://doi.org/10.1136/bmjopen-2015-forum2015...
36517       https://doi.org/10.1089/acm.2014.5277.abstract
37570     https://doi.org/10.5465/ambpp.2015.13255abstract
38444     https://doi.org/10.5465/ambpp.2016.13350abstract
40284    https://doi.org/10.1136/gutjnl-2019-iddfabstra...
45702     https://doi.org/10.5465/ambpp.2022.13625abstract
50384    https://doi.org/10.5465/amproc.2024.18920abstract
52079    https://doi.org/10.1164/ajrccm.2025.211.abstra...
52080    https://doi.org/10.1164/ajrccm.2025.211.abstra...
53602    https://doi.org/10.1158/1538-7755.asgcr25-abst...
Name: doi, dtype: str
0           https://doi.org/10.1016/j.rser.2016.05.022
1          https://doi.org/10.1016/j.agwat.2005.07.001
2         https://doi.

In [81]:
doi= "https://doi.org/10.20659/jfp.16.special_issue_79"
record = df_cr[df_os["DOI"].str.contains(doi, case=False, na=False)]


KeyError: 'DOI'